In [1]:
# !pip install ultralytics
# !pip install roboflow
# !pip install numpy==1.26.4

In [2]:
from ultralytics import YOLO

model = YOLO('yolov8x')
results = model.predict("/kaggle/input/football-clip/challenge-1140_1.mp4", save=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/67) /kaggle/input/football-clip/challenge-1140_1.mp4: 384x640 23 persons, 1 sports ball, 65.2ms
video 1/1 (frame 2/67) /kaggle/i

In [3]:
print(len(results))
print(results[0])

67
ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plan

In [4]:
for box in results[0].boxes:
    print(box)
    break

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([0.], device='cuda:0')
conf: tensor([0.7924], device='cuda:0')
data: tensor([[1.4538e+03, 5.4855e+02, 1.4824e+03, 6.1306e+02, 7.9242e-01, 0.0000e+00]], device='cuda:0')
id: None
is_track: False
orig_shape: (1080, 1920)
shape: torch.Size([1, 6])
xywh: tensor([[1468.0894,  580.8041,   28.6417,   64.5029]], device='cuda:0')
xywhn: tensor([[0.7646, 0.5378, 0.0149, 0.0597]], device='cuda:0')
xyxy: tensor([[1453.7684,  548.5527, 1482.4102,  613.0555]], device='cuda:0')
xyxyn: tensor([[0.7572, 0.5079, 0.7721, 0.5676]], device='cuda:0')


## Football training - Get dataset

In [5]:
import roboflow
roboflow.__version__

'1.2.11'

In [6]:
from roboflow import Roboflow
rf = Roboflow(api_key="ZeYwHkiSLBueVnaZ2Uje")
project = rf.workspace("minaehyeon").project("football-player-jrjtj")
version = project.version(5)
dataset = version.download("yolov5")                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to football-player-5 in yolov5pytorch:: 100%|██████████| 1340/1340 [00:00<00:00, 6968.52it/s]


In [8]:
with open("/kaggle/working/football-player-5/data.yaml") as f:
    print(f.read())

names:
- ball
- goalkeeper
- player
- referee
nc: 4
roboflow:
  license: CC BY 4.0
  project: football-player-jrjtj
  url: https://universe.roboflow.com/minaehyeon/football-player-jrjtj/dataset/5
  version: 5
  workspace: minaehyeon
test: ../test/images
train: football-player-5/train/images
val: football-player-5/valid/images



In [9]:
dataset.location

'/kaggle/working/football-player-5'

In [10]:
# solving dataset path confict
with open("/kaggle/working/football-player-5/data.yaml", "w") as f:
    f.write(
        "names:\n"
        "- ball\n"
        "- goalkeeper\n"
        "- player\n"
        "- referee\n\n"
        "nc: 4\n\n"
        "roboflow:\n"
          "license: CC BY 4.0\n"
          "project: football-player-jrjtj\n"
          "url: https://universe.roboflow.com/minaehyeon/football-player-jrjtj/dataset/5\n"
          "version: 5\n"
          "workspace: minaehyeon\n"
        "train: train/images\n"
        "val: valid/images\n"
        "test: test/images\n"
    )


In [11]:
with open("/kaggle/working/football-player-5/data.yaml") as f:
    print(f.read())

names:
- ball
- goalkeeper
- player
- referee

nc: 4

roboflow:
license: CC BY 4.0
project: football-player-jrjtj
url: https://universe.roboflow.com/minaehyeon/football-player-jrjtj/dataset/5
version: 5
workspace: minaehyeon
train: train/images
val: valid/images
test: test/images



In [14]:
# training yolo v5l model
!yolo task=detect mode=train model=yolov5lu.pt data={dataset.location}/data.yaml epochs=10 imgsz=640

Ultralytics 8.3.239 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/football-player-5/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov5lu.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspe

In [13]:
print(f"{dataset.location}/data.yaml")

/kaggle/working/football-player-5/data.yaml
